# Chapter 10: The chapter studies the symplectomorphism group, Hamiltonian subgroup, flux homomorphism, Calabi homomorphism, and topology of these groups

## Source Span And Standalone Goal

This notebook covers printed pages 385-416 and PDF pages 398-429 of *Introduction to Symplectic Topology*, third edition. The source span was read for structure, terminology, and topic coverage; this notebook is original teaching material and does not copy textbook prose, exercise text, screenshots, or page crops.

**Standalone goal.** Separate symplectic, Hamiltonian, flux, and Calabi information in families of maps.

The chapter focus is: The chapter studies the symplectomorphism group, Hamiltonian subgroup, flux homomorphism, Calabi homomorphism, and topology of these groups. The sections used for orientation are 10.1 Basic properties, 10.2 Flux homomorphism, 10.3 Calabi homomorphism, 10.4 Topology of symplectomorphism groups. A reader should be able to use this notebook without the PDF open: definitions are restated in computational language, examples are small enough to inspect, and every visual is paired with a check rather than treated as decoration.

Keywords for navigation: symplectomorphism group, Hamiltonian group, flux homomorphism, Calabi homomorphism, topology.


## Translation Guide

- A path of symplectomorphisms sweeps area through cycles.
- Flux records that swept cohomology class.
- Calabi integrates compactly supported Hamiltonians into a homomorphism in exact settings.

The recurring move in this course is to turn an abstract symplectic claim into a finite object that can be tested without pretending the finite model proves the full theorem. A closed nondegenerate two-form becomes a matrix or determinant check. A Hamiltonian system becomes a vector field whose flow should preserve energy or signed area. A quotient, fibration, or capacity becomes a picture with a recorded invariant. The aim is not to reduce symplectic topology to plotting; the aim is to make the definitions rigid enough that later theorems have something visible to push against.

A common pitfall for this unit is to read formulas as if they are only coordinate algebra. The notebook instead asks what the formula preserves. When the object is a form, inspect what it pairs. When the object is a map, inspect the residual that says whether the map preserves the form. When the object is an invariant, inspect what changes in the model while the invariant stays fixed.


## Route Through The Unit

- Represent torus translations and Hamiltonian shears.
- Compute flux vectors and compare exact versus non-exact isotopies.
- Use a subgroup diagram to track inclusions and invariants.

The visual spine for this notebook is **flux and subgroup structure**. It is represented as torus strip sweep with subgroup diagram. The learner inspection target is: which isotopies sweep nonzero area through the fundamental cycles. The invariant recorded in the artifact payload is: translation flux equals the swept vector while the Hamiltonian shear has zero average flux.

The representation is deliberately modest and inspectable: NumPy carries the finite-dimensional model, Matplotlib or Plotly renders the artifact, NetworkX is used when the concept is a dependency graph, and JSON records the invariant. This keeps the computational object close to the mathematics rather than hiding the lesson behind a large simulation.

This visual/check pair is a teaching scaffold. It should be read beside the surrounding prose: first identify the mathematical object, then run the check, then inspect the generated artifact under `artifacts/chapter-10-group-of-symplectomorphisms/`. If the check fails after an edit, the failure is part of the lesson because it points to the exact preservation law that was broken.


In [ ]:

from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import networkx as nx
import numpy as np

BOOK_ROOT = Path.cwd()
for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / 'AGENTS.md').exists() and (candidate / 'utils').exists():
        BOOK_ROOT = candidate
        break
else:
    raise RuntimeError('Could not locate the symplectic course root')

if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import ARTIFACT_ROOT, display_artifact
from utils.course_data import unit_by_slug
from utils.symplectic_core import (
    finite_difference,
    harmonic_flow,
    moment_map_c2,
    oscillator_energy,
    polygon_signed_area,
    smooth_step,
    standard_j,
    symplectic_residual,
    wrap_angle,
)
from utils.symplectic_visuals import build_unit_artifacts

UNIT_SLUG = 'chapter-10-group-of-symplectomorphisms'
unit = unit_by_slug(UNIT_SLUG)
np.set_printoptions(precision=4, suppress=True)
print(f'Book root: {BOOK_ROOT}')
print(f'Unit: {unit.label} | printed pages {unit.printed_span} | PDF pages {unit.pdf_span}')


In [ ]:

source_record = unit.to_dict()
print(json.dumps({
    'label': source_record['label'],
    'printed_pages': source_record['printed_pages'],
    'pdf_pages': source_record['pdf_pages'],
    'sections': source_record['sections'],
    'visual_concept': source_record['visual_concept'],
}, indent=2))
assert source_record['printed_pages']
assert source_record['pdf_pages']
assert source_record['source_policy'] if 'source_policy' in source_record else True


## Executable Local Check

The next cell isolates the unit's preservation law before any figure is generated. This makes the artifact honest: the picture is backed by a numerical or structural claim that can fail loudly.

In [ ]:

x = np.linspace(0, 1, 500)
shear = 0.18*np.sin(2*np.pi*x)
translation = np.array([0.28, 0.16])
mean_shear = float(np.trapz(shear, x))
print({'translation_flux_norm': float(np.linalg.norm(translation)), 'mean_shear': mean_shear})
assert float(np.linalg.norm(translation)) > 0
assert abs(mean_shear) < 1e-4


## Generated Artifact

The builder below saves the unit-specific figure or interactive HTML under the course-local `artifacts/` tree, then writes the invariant payload. The displayed image is not a textbook crop; it is generated from the code in this course.

In [ ]:

result = build_unit_artifacts(UNIT_SLUG, ARTIFACT_ROOT)
print(json.dumps(result['assertions'], indent=2, sort_keys=True))
assert all(result['assertions'].values()), result['assertions']
for item in result['artifacts']:
    path = BOOK_ROOT / item['relative_path']
    display_artifact(path, width=760)
    assert path.exists(), item['relative_path']
    assert path.stat().st_size > 200, item['relative_path']


In [ ]:

final_path = BOOK_ROOT / result['final_sanity']
final_payload = json.loads(final_path.read_text(encoding='utf-8'))
print(json.dumps(final_payload, indent=2, sort_keys=True))
assert final_payload['artifact_count'] >= 1
assert all(final_payload['assertions'].values())
for rel_path in final_payload['visual_artifacts']:
    artifact_path = BOOK_ROOT / rel_path
    assert artifact_path.exists(), rel_path
    assert artifact_path.stat().st_size > 200, rel_path
assert final_path.name == 'final-sanity.json'


## Lab Prompt And Takeaways

Lab prompt: Set the translation vector to zero and check that the remaining shear has zero flux.

Before moving on, name the object, the transformation, and the invariant in one sentence. For this unit the object is not just a formula from the page; it is an executable model with a saved figure and a JSON sanity record. That habit matters later when the book moves from local normal forms to global rigidity, fixed-point counts, symplectic embeddings, and open problems.

Takeaways:

- The source span supplies the mathematical agenda; this notebook supplies original explanations and reproducible checks.
- The main visual is useful only because its parameters and invariant are explicit.
- The final `final-sanity.json` file is a small contract: it records which artifacts were made and which claims the computation verified.


In [ ]:

lab_note = {
    'unit': UNIT_SLUG,
    'lab_prompt': 'Set the translation vector to zero and check that the remaining shear has zero flux.',
    'inspection_target': 'which isotopies sweep nonzero area through the fundamental cycles',
    'invariant': 'translation flux equals the swept vector while the Hamiltonian shear has zero average flux',
}
print(json.dumps(lab_note, indent=2))
assert lab_note['unit'] == UNIT_SLUG
